# Notebook 12: Hands-On Lab — SVM Visualization & Text Classification

**Difficulty:** ⭐⭐⭐⭐ | **Estimated time:** 4 hours  
**Theme:** Email / Text Classification

---

## Real-World Analogy First

Imagine you are a postal worker sorting physical letters. Some envelopes are clearly addressed — you can read the label without hesitation. Others are borderline: smudged ink, ambiguous zip code, torn corner. These borderline envelopes are your **support vectors** — the critical, uncertain cases that define your decision procedure.

An SVM is like the expert postal worker who has memorized only these borderline letters. Remove the obvious ones (90% of the pile) and the expert still sorts new letters identically. This is the SVM's radical sparsity property: the entire decision boundary is determined by a tiny subset of training points.

In this lab, we will SEE this property geometrically, extend it to text classification of newsgroup posts, and tune the models through systematic grid search.

---

## Lab Structure

| Section | Topic | Cells |
|---|---|---|
| 0 | Setup & Data Loading | 2 |
| 1 | Hard-Margin SVM from Scratch (QP) | 4 |
| 2 | Soft-Margin SVM Visualization | 4 |
| 3 | Kernel Decision Boundaries | 3 |
| 4 | 20 Newsgroups Text Classification | 5 |
| 5 | Grid Search over C and gamma | 3 |
| 6 | Linear vs. RBF — Decision Boundary Analysis | 3 |
| 7 | Self-Check | 1 |

## Section 0: Setup & Data Loading

In [ ]:
# ── Cell 0.1: Imports ─────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import seaborn as sns
import scipy.optimize
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.svm import SVC, LinearSVC
from sklearn.datasets import make_blobs, make_moons, make_classification, fetch_20newsgroups
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

np.random.seed(42)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

print('All imports successful.')

In [ ]:
# ── Cell 0.2: Dataset Loading & EDA ──────────────────────────────────────

# Part A: 2D geometric data for visualization (linearly separable)
np.random.seed(42)
X_geo, y_geo = make_blobs(n_samples=200, centers=2, cluster_std=0.85,
                           random_state=42, n_features=2)
y_geo = np.where(y_geo == 0, -1, 1)   # SVM uses ±1 labels

# Ensure linear separability: shift centers apart if needed
X_geo[y_geo == -1] -= 1.0
X_geo[y_geo ==  1] += 1.0

X_geo_tr, X_geo_te, y_geo_tr, y_geo_te = train_test_split(
    X_geo, y_geo, test_size=0.25, random_state=42
)
sc_geo = StandardScaler()
X_geo_tr_sc = sc_geo.fit_transform(X_geo_tr)
X_geo_te_sc = sc_geo.transform(X_geo_te)

print(f'Part A (geometric): {X_geo.shape[0]} samples, 2 features, balanced={np.bincount(y_geo==1)}')

# Part B: 20 Newsgroups
categories = ['sci.med', 'sci.space', 'comp.graphics', 'rec.sport.hockey']
print('\nFetching 20 Newsgroups dataset (4 categories)...')
train_data = fetch_20newsgroups(subset='train', categories=categories,
                                 remove=('headers', 'footers', 'quotes'))
test_data  = fetch_20newsgroups(subset='test',  categories=categories,
                                 remove=('headers', 'footers', 'quotes'))

print(f'Train docs: {len(train_data.data)} | Test docs: {len(test_data.data)}')
print(f'Categories: {categories}')

# EDA: class distribution
train_counts = pd.Series(train_data.target).value_counts().sort_index()
cat_labels   = [categories[i].split('.')[-1] for i in train_counts.index]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(cat_labels, train_counts.values,
              color=sns.color_palette('Set2', 4), edgecolor='k')
ax.set_xlabel('Category')
ax.set_ylabel('Number of Documents')
ax.set_title('20 Newsgroups: Training Set Class Distribution\n(4 categories)',
             fontweight='bold')
for bar, val in zip(bars, train_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 5, str(val),
            ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

## Section 1: Hard-Margin SVM from Scratch (QP Solver)

We implement the hard-margin SVM primal formulation using `scipy.optimize.minimize` with SLSQP (Sequential Least Squares Programming), a QP-capable solver.

**Primal problem:**
$$\min_{w, b} \frac{1}{2}||w||^2 \quad \text{s.t.} \quad y_i(w \cdot x_i + b) \geq 1 \quad \forall i$$

The solution is exact: we find the maximum-margin separating hyperplane, and the points satisfying $y_i(w \cdot x_i + b) = 1$ are the **support vectors**.

In [ ]:
# ── Cell 1.1: Hard-Margin SVM via scipy SLSQP ────────────────────────────

def hard_margin_svm_qp(X, y):
    """Hard-margin SVM via scipy SLSQP QP solver.

    Solves: min (1/2)||w||^2  s.t.  y_i(w·x_i + b) >= 1  for all i

    Returns
    -------
    w : ndarray of shape (n_features,)
    b : float
    """
    n, p = X.shape

    def objective(params):
        w = params[:p]
        return 0.5 * np.dot(w, w)

    def grad(params):
        g = np.zeros_like(params)
        g[:p] = params[:p]   # gradient w.r.t. w; b has no gradient in objective
        return g

    # Margin constraints: y_i * (w·x_i + b) - 1 >= 0
    constraints = [
        {'type': 'ineq',
         'fun': lambda params, i=i: y[i] * (X[i] @ params[:p] + params[p]) - 1}
        for i in range(n)
    ]

    x0  = np.zeros(p + 1)
    res = scipy.optimize.minimize(
        objective, x0, jac=grad, method='SLSQP',
        constraints=constraints,
        options={'ftol': 1e-9, 'maxiter': 2000}
    )

    if not res.success:
        print(f'Optimizer warning: {res.message}')

    w = res.x[:p]
    b = res.x[p]
    return w, b


# Use scaled 2D data (hard margin requires separable data)
w_qp, b_qp = hard_margin_svm_qp(X_geo_tr_sc, y_geo_tr)
margin_width = 2.0 / np.linalg.norm(w_qp)

print(f'QP solution:  w = [{w_qp[0]:.4f}, {w_qp[1]:.4f}],  b = {b_qp:.4f}')
print(f'Margin width: 2/||w|| = {margin_width:.4f}')

# Identify support vectors: points with y_i(w·x_i + b) ≈ 1
margins_i = y_geo_tr * (X_geo_tr_sc @ w_qp + b_qp)
sv_mask   = np.abs(margins_i - 1.0) < 0.05   # tolerance
sv_indices = np.where(sv_mask)[0]
print(f'\nSupport vectors found: {sv_mask.sum()} out of {len(y_geo_tr)} training points')
print(f'SV indices: {sv_indices}')
print(f'SV coordinates:\n{X_geo_tr_sc[sv_mask]}')

In [ ]:
# ── Cell 1.2: Visualize + Verify vs sklearn ────────────────────────────────

def plot_svm_2d(X, y, w, b, sv_mask=None, ax=None, title='SVM Decision Boundary'):
    """Plot 2D SVM decision boundary, margin lines, and support vectors."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 6))

    x1_min, x1_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    x2_min, x2_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx = np.linspace(x1_min, x1_max, 400)

    # Decision boundary: w[0]*x1 + w[1]*x2 + b = 0  →  x2 = -(w[0]*x1 + b)/w[1]
    if abs(w[1]) > 1e-8:
        x2_boundary = -(w[0] * xx + b) / w[1]
        x2_margin_p = -(w[0] * xx + b - 1) / w[1]
        x2_margin_n = -(w[0] * xx + b + 1) / w[1]
        ax.plot(xx, x2_boundary, 'k-',  linewidth=2.5, label='Decision boundary')
        ax.plot(xx, x2_margin_p, 'k--', linewidth=1.5, label='Margin (w·x+b=+1)')
        ax.plot(xx, x2_margin_n, 'k:',  linewidth=1.5, label='Margin (w·x+b=-1)')
        ax.fill_between(xx, x2_margin_n, x2_margin_p, alpha=0.1, color='gray',
                         label='Margin band')

    # Scatter plot
    colors = {-1: '#e74c3c', 1: '#3498db'}
    labels = {-1: 'Class -1', 1: 'Class +1'}
    for cls in [-1, 1]:
        mask = y == cls
        ax.scatter(X[mask, 0], X[mask, 1], c=colors[cls],
                   edgecolors='k', linewidths=0.5, s=50,
                   label=labels[cls], zorder=3)

    # Highlight SVs
    if sv_mask is not None and sv_mask.any():
        ax.scatter(X[sv_mask, 0], X[sv_mask, 1],
                   s=200, facecolors='none', edgecolors='gold',
                   linewidths=2.5, label=f'Support Vectors ({sv_mask.sum()})', zorder=4)

    ax.set_xlim(x1_min, x1_max)
    ax.set_ylim(x2_min, x2_max)
    ax.set_xlabel('Feature 1 (scaled)')
    ax.set_ylabel('Feature 2 (scaled)')
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=8, loc='upper left')
    return ax


# sklearn hard-margin SVM (C=1e6 approximates hard margin)
svc_hardmargin = SVC(kernel='linear', C=1e6)
svc_hardmargin.fit(X_geo_tr_sc, y_geo_tr)
w_sk = svc_hardmargin.coef_[0]
b_sk = svc_hardmargin.intercept_[0]
sv_mask_sk = np.zeros(len(X_geo_tr_sc), dtype=bool)
for sv in svc_hardmargin.support_vectors_:
    dists = np.linalg.norm(X_geo_tr_sc - sv, axis=1)
    sv_mask_sk[np.argmin(dists)] = True

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

plot_svm_2d(X_geo_tr_sc, y_geo_tr, w_qp, b_qp, sv_mask,
            ax=axes[0], title=f'QP Solver (scipy SLSQP)\n'
                               f'w=[{w_qp[0]:.3f},{w_qp[1]:.3f}], b={b_qp:.3f}\n'
                               f'Margin={margin_width:.3f}')

plot_svm_2d(X_geo_tr_sc, y_geo_tr, w_sk, b_sk, sv_mask_sk,
            ax=axes[1], title=f'sklearn SVC(kernel=linear, C=1e6)\n'
                               f'w=[{w_sk[0]:.3f},{w_sk[1]:.3f}], b={b_sk:.3f}\n'
                               f'Margin={2/np.linalg.norm(w_sk):.3f}')

plt.suptitle('Hard-Margin SVM: QP Solver vs. sklearn\nBoundaries should be nearly identical',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nQP    w: [{w_qp[0]:.4f}, {w_qp[1]:.4f}], b: {b_qp:.4f}')
print(f'sklearn w: [{w_sk[0]:.4f}, {w_sk[1]:.4f}], b: {b_sk:.4f}')
print(f'\nNote: sign may differ (w,-b) vs (-w,b) — only direction matters.')

In [ ]:
# ── Cell 1.3: Sparsity — Deletion Test ────────────────────────────────────
# The key SVM property: non-SVs are irrelevant to the boundary.
# Retrain on SVs only → identical boundary.

# Get support vector indices from sklearn (exact)
sv_idx_exact = svc_hardmargin.support_
print(f'sklearn support vector indices: {sv_idx_exact}')
print(f'Number of support vectors: {len(sv_idx_exact)} / {len(X_geo_tr_sc)} training points')
print(f'Sparsity: only {len(sv_idx_exact)/len(X_geo_tr_sc):.1%} of training data defines the boundary!')

# Retain ONLY support vectors
X_sv_only = X_geo_tr_sc[sv_idx_exact]
y_sv_only = y_geo_tr[sv_idx_exact]

# Retrain on SVs only
svc_sv_only = SVC(kernel='linear', C=1e6)
svc_sv_only.fit(X_sv_only, y_sv_only)
w_sv = svc_sv_only.coef_[0]
b_sv = svc_sv_only.intercept_[0]

# Compare on test set
acc_full = accuracy_score(y_geo_te, svc_hardmargin.predict(X_geo_te_sc))
acc_sv   = accuracy_score(y_geo_te, svc_sv_only.predict(X_geo_te_sc))

print(f'\nFull dataset  → w=[{w_sk[0]:.4f},{w_sk[1]:.4f}], b={b_sk:.4f}, acc={acc_full:.4f}')
print(f'SVs only      → w=[{w_sv[0]:.4f},{w_sv[1]:.4f}], b={b_sv:.4f}, acc={acc_sv:.4f}')

# Normalize w for direction comparison
w_sk_n = w_sk / np.linalg.norm(w_sk)
w_sv_n = w_sv / np.linalg.norm(w_sv)
cosine_sim = abs(np.dot(w_sk_n, w_sv_n))
print(f'\nCosine similarity of weight vectors: {cosine_sim:.6f}')
print('→ 1.0 means identical direction (boundary is the same)')

if acc_full == acc_sv:
    print('\nDeletion test PASSED: Removing non-SVs gives identical accuracy.')
else:
    print(f'\nNote: small numerical difference — margins may differ slightly due to small SV count.')

In [ ]:
# ── Cell 1.4: Visual Confirmation of Deletion Test ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: full dataset with SV highlighted
sv_mask_full = np.zeros(len(X_geo_tr_sc), dtype=bool)
sv_mask_full[sv_idx_exact] = True
plot_svm_2d(X_geo_tr_sc, y_geo_tr, w_sk, b_sk, sv_mask_full,
            ax=axes[0],
            title=f'Full Training Set ({len(X_geo_tr_sc)} points)\n'
                   f'{len(sv_idx_exact)} SVs highlighted in gold\nAcc={acc_full:.4f}')

# Right: SV-only dataset
sv_mask_all = np.ones(len(X_sv_only), dtype=bool)  # all are SVs
plot_svm_2d(X_sv_only, y_sv_only, w_sv, b_sv, sv_mask_all,
            ax=axes[1],
            title=f'SV-Only Training ({len(X_sv_only)} points)\n'
                   f'Non-SVs deleted — boundary unchanged!\nAcc={acc_sv:.4f}')

plt.suptitle('SVM Sparsity: Deletion Test\n'
             'Removing non-SVs does not change the decision boundary',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nKey insight: The SVM only "remembers" the support vectors.')
print('This is what makes SVM memory-efficient at test time.')

## Section 2: Soft-Margin SVM Visualization

Real data is rarely linearly separable. The soft-margin SVM introduces slack variables $\xi_i \geq 0$:

$$\min_{w,b,\xi} \frac{1}{2}||w||^2 + C \sum_i \xi_i \quad \text{s.t.} \quad y_i(w \cdot x_i + b) \geq 1 - \xi_i$$

- **Large C**: small tolerance for misclassification → narrow margin, few SVs (risks overfitting)
- **Small C**: large tolerance → wide margin, many SVs (risks underfitting)

C is the most important SVM hyperparameter.

In [ ]:
# ── Cell 2.1: Non-Separable Data + C Sweep ────────────────────────────────
np.random.seed(42)

# Generate overlapping 2D data (non-linearly separable)
X_soft, y_soft = make_classification(
    n_samples=300, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, class_sep=0.8, random_state=42
)
y_soft = np.where(y_soft == 0, -1, 1)

sc_soft = StandardScaler()
X_soft_sc = sc_soft.fit_transform(X_soft)

C_values = [0.01, 0.1, 1.0, 10.0]
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

def plot_soft_margin(X, y, C, ax):
    clf = SVC(kernel='linear', C=C)
    clf.fit(X, y)

    x1 = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 400)
    w   = clf.coef_[0]
    b   = clf.intercept_[0]
    margin = 2.0 / np.linalg.norm(w)
    n_sv = len(clf.support_)

    if abs(w[1]) > 1e-8:
        x2_bd  = -(w[0]*x1 + b) / w[1]
        x2_mp  = -(w[0]*x1 + b - 1) / w[1]
        x2_mn  = -(w[0]*x1 + b + 1) / w[1]
        ax.plot(x1, x2_bd,  'k-',  lw=2.5, label='Boundary')
        ax.plot(x1, x2_mp,  'b--', lw=1.5, label='Margin +1')
        ax.plot(x1, x2_mn,  'r--', lw=1.5, label='Margin -1')
        ax.fill_between(x1, x2_mn, x2_mp, alpha=0.08, color='gray')

    for cls, color in [(-1, '#e74c3c'), (1, '#3498db')]:
        m = y == cls
        ax.scatter(X[m, 0], X[m, 1], c=color, edgecolors='k',
                   linewidths=0.4, s=40, alpha=0.8)

    # Highlight SVs with circles
    ax.scatter(X[clf.support_, 0], X[clf.support_, 1],
               s=160, facecolors='none', edgecolors='gold', linewidths=2,
               label=f'SVs: {n_sv}', zorder=5)

    acc = accuracy_score(y, clf.predict(X))
    ax.set_title(f'C = {C}  |  Margin width = {margin:.3f}\n'
                  f'Support Vectors: {n_sv}  |  Train Acc: {acc:.3f}',
                  fontweight='bold')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.legend(fontsize=8, loc='upper left')
    ax.set_xlim(X[:, 0].min()-0.5, X[:, 0].max()+0.5)
    ax.set_ylim(X[:, 1].min()-0.5, X[:, 1].max()+0.5)

for ax, C in zip(axes, C_values):
    plot_soft_margin(X_soft_sc, y_soft, C, ax)

plt.suptitle('Soft-Margin SVM: Effect of C on Margin Width and Support Vectors\n'
             'C=0.01 → wide margin, many SVs (underfitting) | C=10 → narrow margin, few SVs (overfitting)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 2.2: C vs. Margin Width & SV Count Table ─────────────────────────
rows = []
for C in [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0]:
    clf = SVC(kernel='linear', C=C)
    clf.fit(X_soft_sc, y_soft)
    w   = clf.coef_[0]
    margin = 2.0 / np.linalg.norm(w)
    rows.append({'C': C, 'Margin Width': round(margin, 4),
                  'Num SVs': len(clf.support_),
                  'Train Acc': round(accuracy_score(y_soft, clf.predict(X_soft_sc)), 4)})

df_c = pd.DataFrame(rows).set_index('C')
print(df_c.to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].semilogx(df_c.index, df_c['Margin Width'], 'o-', color='#3498db', lw=2)
axes[0].set_xlabel('C (log scale)'); axes[0].set_ylabel('Margin Width')
axes[0].set_title('C vs. Margin Width\n(larger C = narrower margin)', fontweight='bold')

axes[1].semilogx(df_c.index, df_c['Num SVs'], 's-', color='#e74c3c', lw=2)
axes[1].set_xlabel('C (log scale)'); axes[1].set_ylabel('Number of SVs')
axes[1].set_title('C vs. Number of SVs\n(larger C = fewer SVs)', fontweight='bold')

axes[2].semilogx(df_c.index, df_c['Train Acc'], 'D-', color='#27ae60', lw=2)
axes[2].set_xlabel('C (log scale)'); axes[2].set_ylabel('Train Accuracy')
axes[2].set_title('C vs. Train Accuracy\n(diminishing returns at high C)', fontweight='bold')

plt.suptitle('Effect of Regularization Parameter C on Linear SVM',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 2.3: Effect of Outliers on Soft-Margin SVM ──────────────────────
np.random.seed(42)

# Add 3 deliberate outliers near the wrong class
X_out = X_soft_sc.copy()
y_out = y_soft.copy()

outliers_X = np.array([[2.5, -2.0], [-2.5, 2.0], [1.8, -1.5]])
outliers_y = np.array([-1, 1, -1])   # wrong side of margin

X_with_out = np.vstack([X_out, outliers_X])
y_with_out = np.concatenate([y_out, outliers_y])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Low C: tolerates outliers, boundary barely moves
plot_soft_margin(X_with_out, y_with_out, C=0.1, ax=axes[0])
axes[0].scatter(outliers_X[:, 0], outliers_X[:, 1],
                marker='X', s=200, color='orange', edgecolors='k', linewidths=1.5,
                label='Outliers', zorder=6)
axes[0].set_title('C=0.1 (Small): Wide margin tolerates outliers\n'
                   'Boundary robust to noise', fontweight='bold')
axes[0].legend(fontsize=8)

# High C: forced to accommodate outliers → boundary distorted
plot_soft_margin(X_with_out, y_with_out, C=10.0, ax=axes[1])
axes[1].scatter(outliers_X[:, 0], outliers_X[:, 1],
                marker='X', s=200, color='orange', edgecolors='k', linewidths=1.5,
                label='Outliers', zorder=6)
axes[1].set_title('C=10 (Large): Narrow margin, outliers become SVs\n'
                   'Boundary potentially overfits to noise', fontweight='bold')
axes[1].legend(fontsize=8)

plt.suptitle('Outlier Effect on Soft-Margin SVM\nX markers = deliberate outliers',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Key insight: Small C = the SVM is robust to outliers (wide margin absorbs noise).')
print('Large C = outliers become support vectors and pull the boundary toward them.')

In [ ]:
# ── Cell 2.4: Observations Summary ───────────────────────────────────────
print('Soft-Margin SVM: Key Observations')
print('=' * 50)
print()
print('C (Regularization)  | Margin Width | SVs  | Behavior')
print('-' * 65)
for C, row in df_c.iterrows():
    behavior = 'underfitting' if C <= 0.01 else ('well-tuned' if C <= 5.0 else 'risk overfit')
    print(f'C={C:<8}          | {row["Margin Width"]:<12.4f} | {row["Num SVs"]:<5} | {behavior}')

print()
print('Takeaway:')
print('  - Larger C → fewer SVs, narrower margin, higher train accuracy')
print('  - Smaller C → more SVs, wider margin, more regularized')
print('  - Optimal C is found via cross-validation (GridSearchCV)')

## Section 3: Kernel Decision Boundaries

The **kernel trick** lets SVMs find non-linear boundaries without explicitly computing the feature transformation. The kernel function $K(x_i, x_j) = \phi(x_i) \cdot \phi(x_j)$ measures similarity in a high-dimensional feature space.

| Kernel | Formula | When to use |
|---|---|---|
| Linear | $x_i \cdot x_j$ | Text, high-dim, large n |
| Polynomial | $(x_i \cdot x_j + r)^d$ | Image features, structured data |
| RBF (Gaussian) | $\exp(-\gamma||x_i-x_j||^2)$ | Most non-linear problems |

For RBF: **$\gamma$** controls the "reach" of each training point. High $\gamma$ = each point only influences its immediate neighborhood → complex, potentially overfit boundary.

In [ ]:
# ── Cell 3.1: XOR Dataset + 6-Kernel Comparison ──────────────────────────
np.random.seed(42)

# XOR-like dataset: 4 clusters in ± pattern
n_per_cluster = 50
X_xor = np.vstack([
    np.random.randn(n_per_cluster, 2) + [ 1.5,  1.5],  # class +1
    np.random.randn(n_per_cluster, 2) + [-1.5, -1.5],  # class +1
    np.random.randn(n_per_cluster, 2) + [ 1.5, -1.5],  # class -1
    np.random.randn(n_per_cluster, 2) + [-1.5,  1.5],  # class -1
])
y_xor = np.array([1]*n_per_cluster + [1]*n_per_cluster +
                  [-1]*n_per_cluster + [-1]*n_per_cluster)

sc_xor = StandardScaler()
X_xor_sc = sc_xor.fit_transform(X_xor)

kernels = [
    ('Linear',         SVC(kernel='linear',  C=1.0)),
    ('Poly d=2',       SVC(kernel='poly',     C=1.0, degree=2, gamma='auto')),
    ('Poly d=3',       SVC(kernel='poly',     C=1.0, degree=3, gamma='auto')),
    ('RBF γ=0.1',      SVC(kernel='rbf',      C=1.0, gamma=0.1)),
    ('RBF γ=1.0',      SVC(kernel='rbf',      C=1.0, gamma=1.0)),
    ('RBF γ=10',       SVC(kernel='rbf',      C=1.0, gamma=10.0)),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

x1_range = np.linspace(X_xor_sc[:, 0].min()-0.3, X_xor_sc[:, 0].max()+0.3, 400)
x2_range = np.linspace(X_xor_sc[:, 1].min()-0.3, X_xor_sc[:, 1].max()+0.3, 400)
xx, yy = np.meshgrid(x1_range, x2_range)

cmap_bg   = ListedColormap(['#ffcccc', '#ccffcc'])
cmap_pts  = ListedColormap(['#e74c3c', '#2ecc71'])

for ax, (name, clf) in zip(axes, kernels):
    clf.fit(X_xor_sc, y_xor)
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    Z_num = np.where(Z == -1, 0, 1)

    ax.contourf(xx, yy, Z_num, cmap=cmap_bg, alpha=0.5)
    ax.contour(xx, yy, Z_num, colors='k', linewidths=1.5)

    for cls, color, marker in [(-1, '#e74c3c', 'o'), (1, '#2ecc71', 's')]:
        m = y_xor == cls
        ax.scatter(X_xor_sc[m, 0], X_xor_sc[m, 1], c=color,
                   marker=marker, edgecolors='k', linewidths=0.4, s=40, alpha=0.85)

    acc  = accuracy_score(y_xor, clf.predict(X_xor_sc))
    n_sv = len(clf.support_)
    ax.set_title(f'{name}  |  Acc={acc:.3f}  |  SVs={n_sv}', fontweight='bold')
    ax.set_xlabel('x1'); ax.set_ylabel('x2')

plt.suptitle('Kernel Decision Boundaries on XOR Dataset\n'
             'Linear kernel fails; RBF adapts to non-linear structure',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 3.2: RBF Gamma — Underfitting to Overfitting ────────────────────
gammas = [0.01, 0.05, 0.2, 0.5, 1.0, 3.0, 10.0, 50.0, 100.0]
n_cols = 3
n_rows = (len(gammas) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 13))
axes = axes.ravel()

for ax, gamma in zip(axes, gammas):
    clf = SVC(kernel='rbf', C=1.0, gamma=gamma)
    clf.fit(X_xor_sc, y_xor)
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    Z_num = np.where(Z == -1, 0, 1)

    ax.contourf(xx, yy, Z_num, cmap=cmap_bg, alpha=0.5)
    ax.contour(xx, yy, Z_num, colors='k', linewidths=1.2)

    for cls, color, marker in [(-1, '#e74c3c', 'o'), (1, '#2ecc71', 's')]:
        m = y_xor == cls
        ax.scatter(X_xor_sc[m, 0], X_xor_sc[m, 1], c=color,
                   marker=marker, edgecolors='k', linewidths=0.3, s=30, alpha=0.8)

    acc  = accuracy_score(y_xor, clf.predict(X_xor_sc))
    n_sv = len(clf.support_)
    regime = 'underfit' if gamma < 0.2 else ('overfit' if gamma > 10 else 'good')
    ax.set_title(f'γ={gamma}  |  {regime}\nAcc={acc:.3f}  SVs={n_sv}',
                  fontweight='bold', fontsize=9)
    ax.set_xlabel('x1', fontsize=8); ax.set_ylabel('x2', fontsize=8)

# Hide any unused subplots
for ax in axes[len(gammas):]:
    ax.set_visible(False)

plt.suptitle('RBF Kernel: γ from Underfitting (γ=0.01) to Overfitting (γ=100)\n'
             'γ=0.01: too smooth (underfits XOR) | γ=100: memorizes each point (overfits)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nAs γ increases:')
print('  - Boundary becomes increasingly complex (more local)')
print('  - Each SV exerts influence only over an infinitesimally small radius')
print('  - Gram matrix becomes nearly diagonal → extreme overfitting')

In [ ]:
# ── Cell 3.3: Gamma Effect on Gram Matrix Structure ───────────────────────
# High gamma → Gram matrix nearly diagonal (each point is only similar to itself)

sample_idx = np.random.choice(len(X_xor_sc), 40, replace=False)
X_sample = X_xor_sc[sample_idx]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, gamma in zip(axes, [0.05, 1.0, 50.0]):
    # Compute RBF Gram matrix manually
    dists = np.sum((X_sample[:, None, :] - X_sample[None, :, :])**2, axis=-1)
    K = np.exp(-gamma * dists)
    sns.heatmap(K, ax=ax, cmap='YlOrRd', vmin=0, vmax=1,
                xticklabels=False, yticklabels=False, cbar_kws={'shrink': 0.7})
    ax.set_title(f'γ = {gamma}\nMean off-diag: {(K - np.eye(len(K))).mean():.4f}',
                  fontweight='bold')
    ax.set_xlabel('Sample index')
    ax.set_ylabel('Sample index')

plt.suptitle('RBF Gram (Kernel) Matrix as γ Increases\n'
             'Low γ: smooth similarity | High γ: nearly diagonal (each point isolated)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nWith γ=50: Gram matrix is nearly identity — the SVM sees each point as unique.')
print('This leads to perfect training accuracy but catastrophic test performance.')

## Section 4: 20 Newsgroups — Text Classification

We now apply SVMs to a real-world multi-class text classification problem: the **20 Newsgroups** dataset, using 4 categories:
- `sci.med` — medical discussions
- `sci.space` — space exploration
- `comp.graphics` — computer graphics
- `rec.sport.hockey` — hockey discussions

**Pipeline:** TF-IDF vectorization (10,000 features) → SVM classifier

For RBF SVM, we first reduce dimensionality via TruncatedSVD (100 dims) since RBF doesn't benefit from sparse high-dimensional features the same way.

In [ ]:
# ── Cell 4.1: TF-IDF Pipeline ─────────────────────────────────────────────
# LinearSVC on raw TF-IDF (10K features)
tfidf = TfidfVectorizer(max_features=10000, sublinear_tf=True, min_df=3, ngram_range=(1, 2))

X_train_tfidf = tfidf.fit_transform(train_data.data)
X_test_tfidf  = tfidf.transform(test_data.data)
y_train_ng    = train_data.target
y_test_ng     = test_data.target

print(f'TF-IDF matrix: train={X_train_tfidf.shape}, test={X_test_tfidf.shape}')
print(f'Sparsity: {1 - X_train_tfidf.nnz / (X_train_tfidf.shape[0]*X_train_tfidf.shape[1]):.2%}')

# 1. LinearSVC on TF-IDF
t0 = time.perf_counter()
clf_linear = LinearSVC(C=1.0, max_iter=3000)
clf_linear.fit(X_train_tfidf, y_train_ng)
t_linear = time.perf_counter() - t0
y_pred_linear = clf_linear.predict(X_test_tfidf)

acc_linear = accuracy_score(y_test_ng, y_pred_linear)
f1_linear  = f1_score(y_test_ng, y_pred_linear, average='macro')

print(f'\nLinearSVC (TF-IDF): time={t_linear:.2f}s | acc={acc_linear:.4f} | F1-macro={f1_linear:.4f}')

# 2. RBF SVM on TruncatedSVD (100 dims)
svd = TruncatedSVD(n_components=100, random_state=42)
X_train_svd = svd.fit_transform(X_train_tfidf)
X_test_svd  = svd.transform(X_test_tfidf)

sc_ng = StandardScaler()
X_train_svd_sc = sc_ng.fit_transform(X_train_svd)
X_test_svd_sc  = sc_ng.transform(X_test_svd)

t0 = time.perf_counter()
clf_rbf = SVC(kernel='rbf', C=10.0, gamma='scale')
clf_rbf.fit(X_train_svd_sc, y_train_ng)
t_rbf = time.perf_counter() - t0
y_pred_rbf = clf_rbf.predict(X_test_svd_sc)

acc_rbf = accuracy_score(y_test_ng, y_pred_rbf)
f1_rbf  = f1_score(y_test_ng, y_pred_rbf, average='macro')

print(f'SVC RBF (SVD-100):  time={t_rbf:.2f}s | acc={acc_rbf:.4f} | F1-macro={f1_rbf:.4f}')

df_ng = pd.DataFrame([
    {'Model': 'LinearSVC (TF-IDF 10K)', 'Train Time': t_linear,
      'Accuracy': acc_linear, 'F1-macro': f1_linear},
    {'Model': 'SVC RBF (SVD-100)',       'Train Time': t_rbf,
      'Accuracy': acc_rbf, 'F1-macro': f1_rbf},
]).set_index('Model')
print('\n', df_ng.round(4).to_string())

In [ ]:
# ── Cell 4.2: Top Features per Class (LinearSVC) ──────────────────────────
cat_short = ['comp.graphics', 'rec.sport.hockey', 'sci.med', 'sci.space']
vocab      = np.array(tfidf.get_feature_names_out())
n_top      = 15

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

# Ensure categories match target indices
actual_cats = [train_data.target_names[i] for i in range(len(train_data.target_names))]

for cls_idx, (ax, cat) in enumerate(zip(axes, actual_cats)):
    coefs     = clf_linear.coef_[cls_idx]
    top_idx   = np.argsort(coefs)[-n_top:][::-1]
    top_words = vocab[top_idx]
    top_coefs = coefs[top_idx]

    colors = sns.color_palette('coolwarm_r', n_top)
    bars = ax.barh(range(n_top), top_coefs[::-1], color=colors[::-1])
    ax.set_yticks(range(n_top))
    ax.set_yticklabels(top_words[::-1], fontsize=9)
    ax.set_xlabel('LinearSVC Coefficient')
    ax.set_title(f'Top {n_top} Words: {cat}', fontweight='bold')
    ax.invert_yaxis()

plt.suptitle('LinearSVC: Most Discriminative Words per Category\n'
             '20 Newsgroups (TF-IDF, 10K features)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('The top words are highly interpretable — this is a key advantage of LinearSVC!')

In [ ]:
# ── Cell 4.3: Confusion Matrix Comparison ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

cat_labels_short = ['comp.graph', 'rec.hockey', 'sci.med', 'sci.space']

for ax, (y_pred_i, title) in zip(axes, [
    (y_pred_linear, 'LinearSVC (TF-IDF)'),
    (y_pred_rbf,    'SVC RBF (SVD-100)'),
]):
    cm = confusion_matrix(y_test_ng, y_pred_i, normalize='true')
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', ax=ax,
                xticklabels=cat_labels_short,
                yticklabels=cat_labels_short,
                vmin=0, vmax=1)
    acc_i = accuracy_score(y_test_ng, y_pred_i)
    f1_i  = f1_score(y_test_ng, y_pred_i, average='macro')
    ax.set_title(f'{title}\nAcc={acc_i:.4f} | F1-macro={f1_i:.4f}', fontweight='bold')
    ax.set_xlabel('Predicted Category')
    ax.set_ylabel('True Category')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Confusion Matrices: LinearSVC vs. RBF SVM\n20 Newsgroups (4 categories, normalized)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 4.4: Per-Category Performance Breakdown ──────────────────────────
from sklearn.metrics import precision_recall_fscore_support

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics_names = ['Precision', 'Recall', 'F1']

for ax, metric, idx in zip(axes, metrics_names, range(3)):
    linear_vals = precision_recall_fscore_support(y_test_ng, y_pred_linear, average=None)[idx]
    rbf_vals    = precision_recall_fscore_support(y_test_ng, y_pred_rbf,    average=None)[idx]

    x_pos = np.arange(len(cat_labels_short))
    width = 0.35
    ax.bar(x_pos - width/2, linear_vals, width, label='LinearSVC',  color='#3498db', alpha=0.85)
    ax.bar(x_pos + width/2, rbf_vals,    width, label='SVC (RBF)',   color='#e74c3c', alpha=0.85)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(cat_labels_short, rotation=15, fontsize=9)
    ax.set_ylim(0.5, 1.05)
    ax.set_title(f'{metric} per Category', fontweight='bold')
    ax.set_ylabel(metric)
    ax.legend(fontsize=9)

plt.suptitle('Per-Category Performance: LinearSVC vs. RBF SVM\n'
             'LinearSVC consistently outperforms RBF on this text classification task',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nLinearSVC advantages on text data:')
print('  1. Sparse TF-IDF features: linear boundaries suffice (no complex interactions)')
print('  2. No dimensionality reduction needed → preserves all information')
print('  3. Interpretable: coef_[k] directly maps to word importance')
print('  4. Much faster training: O(n·p) vs O(n²) for kernel SVM')

In [ ]:
# ── Cell 4.5: Sample Documents Near Decision Boundary ─────────────────────
# Find test documents that LinearSVC is least confident about
decision_scores = clf_linear.decision_function(X_test_tfidf)
# Max score minus second-max score: small value = uncertain prediction
sorted_scores = np.sort(decision_scores, axis=1)  # sort columns
margin_score  = sorted_scores[:, -1] - sorted_scores[:, -2]  # winner margin

# Most uncertain predictions
uncertain_idx = np.argsort(margin_score)[:5]

print('5 Most Uncertain Test Documents (smallest decision margin):')
print('=' * 70)
for i, idx in enumerate(uncertain_idx):
    true_cat  = train_data.target_names[y_test_ng[idx]].split('.')[-1]
    pred_cat  = train_data.target_names[y_pred_linear[idx]].split('.')[-1]
    snippet   = test_data.data[idx][:200].replace('\n', ' ').strip()
    correct   = '✓' if y_test_ng[idx] == y_pred_linear[idx] else '✗'
    print(f'\n[{i+1}] True: {true_cat:15s} | Pred: {pred_cat:15s} | Margin: {margin_score[idx]:.4f} {correct}')
    print(f'    "{snippet[:180]}..."')

## Section 5: Grid Search over C and gamma

The RBF SVM has two key hyperparameters:
- **C**: regularization (higher C = harder margin)
- **gamma**: RBF bandwidth (higher gamma = more local, complex boundary)

These interact: high C + high gamma = severe overfitting. We find the optimal combination via 5-fold cross-validation.

In [ ]:
# ── Cell 5.1: GridSearchCV on RBF SVM ─────────────────────────────────────
# Use SVD-reduced 20 Newsgroups (100 dims)
pipe = Pipeline([
    ('svc', SVC(kernel='rbf'))
])

param_grid = {
    'svc__C':     [0.1, 1, 10, 100],
    'svc__gamma': ['scale', 0.01, 0.1, 1.0]
}

print('Running GridSearchCV (5-fold, 4×4=16 configurations)...')
t0 = time.perf_counter()
gs = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=0)
gs.fit(X_train_svd_sc, y_train_ng)
gs_time = time.perf_counter() - t0

print(f'Grid search time: {gs_time:.1f}s')
print(f'Best params: {gs.best_params_}')
print(f'Best CV accuracy: {gs.best_score_:.4f}')

y_pred_best = gs.predict(X_test_svd_sc)
acc_best = accuracy_score(y_test_ng, y_pred_best)
f1_best  = f1_score(y_test_ng, y_pred_best, average='macro')
print(f'Test accuracy (best RBF): {acc_best:.4f} | F1-macro: {f1_best:.4f}')
print(f'LinearSVC test accuracy:  {acc_linear:.4f}')

In [ ]:
# ── Cell 5.2: Grid Search Heatmap ─────────────────────────────────────────
results_df = pd.DataFrame(gs.cv_results_)

# Build pivot table for heatmap
C_vals     = [0.1, 1, 10, 100]
gamma_vals = ['scale', 0.01, 0.1, 1.0]

score_matrix = np.zeros((len(C_vals), len(gamma_vals)))
for i, C in enumerate(C_vals):
    for j, gamma in enumerate(gamma_vals):
        mask = (
            (results_df['param_svc__C'] == C) &
            (results_df['param_svc__gamma'].astype(str) == str(gamma))
        )
        score_matrix[i, j] = results_df.loc[mask, 'mean_test_score'].values[0]

df_heatmap = pd.DataFrame(score_matrix,
                           index=[f'C={c}' for c in C_vals],
                           columns=[f'γ={g}' for g in gamma_vals])

# Find best cell
best_C_idx     = C_vals.index(gs.best_params_['svc__C'])
best_gamma_str = str(gs.best_params_['svc__gamma'])
best_gamma_idx = [str(g) for g in gamma_vals].index(best_gamma_str)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(df_heatmap, annot=True, fmt='.4f', cmap='YlOrRd',
            ax=ax, vmin=score_matrix.min()-0.02, vmax=score_matrix.max()+0.02,
            linewidths=0.5, linecolor='gray')

# Mark best cell with a rectangle
ax.add_patch(plt.Rectangle((best_gamma_idx, best_C_idx), 1, 1,
                              fill=False, edgecolor='blue', lw=3, label='Best params'))

ax.set_title(f'GridSearchCV: 5-Fold CV Accuracy (C × gamma)\n'
              f'Best: C={gs.best_params_["svc__C"]}, γ={gs.best_params_["svc__gamma"]}  '
              f'| CV acc={gs.best_score_:.4f}',
              fontweight='bold')
ax.set_xlabel('gamma')
ax.set_ylabel('C')
plt.tight_layout()
plt.show()

print(f'\nBest RBF SVM test acc: {acc_best:.4f}  vs  LinearSVC: {acc_linear:.4f}')
print(f'Linear kernel advantage: {acc_linear - acc_best:.4f} — text is a linear problem!')

In [ ]:
# ── Cell 5.3: CV Score Distribution ──────────────────────────────────────
all_cv_scores = results_df['mean_test_score'].values
all_params    = [f'C={r["param_svc__C"]},γ={r["param_svc__gamma"]}'
                  for _, r in results_df.iterrows()]

sorted_order  = np.argsort(all_cv_scores)[::-1]

fig, ax = plt.subplots(figsize=(14, 5))
colors = ['#27ae60' if i == 0 else '#3498db' for i in range(len(all_cv_scores))]
ax.bar(range(len(all_cv_scores)),
        all_cv_scores[sorted_order],
        color=['#27ae60'] + ['#3498db']*(len(all_cv_scores)-1))
ax.set_xticks(range(len(all_cv_scores)))
ax.set_xticklabels([all_params[i] for i in sorted_order],
                    rotation=45, ha='right', fontsize=8)
ax.axhline(acc_linear, color='red', linestyle='--', linewidth=2,
            label=f'LinearSVC baseline ({acc_linear:.4f})')
ax.set_ylabel('5-Fold CV Accuracy')
ax.set_title('Grid Search Results: All 16 Configurations Ranked\n'
              'Red dashed line = LinearSVC baseline (hard to beat on text!)', fontweight='bold')
ax.legend()
ax.set_ylim(all_cv_scores.min() - 0.02, 1.02)
plt.tight_layout()
plt.show()

## Section 6: Linear vs. RBF — Decision Boundary Analysis on 2D Projection

To visualize decision boundaries for text data, we reduce 20 Newsgroups to 2D using TruncatedSVD. This is a lossy projection, so boundaries here are illustrative — the real classifier operates in 10,000 dimensions.

The 2D visualization reveals **why linear works better**: TF-IDF feature distributions across categories are largely linearly separable in the latent space.

In [ ]:
# ── Cell 6.1: 2D Projection + Decision Boundaries ─────────────────────────
svd2 = TruncatedSVD(n_components=2, random_state=42)
X_train_2d = svd2.fit_transform(X_train_tfidf)
X_test_2d  = svd2.transform(X_test_tfidf)

sc2d = StandardScaler()
X_train_2d_sc = sc2d.fit_transform(X_train_2d)
X_test_2d_sc  = sc2d.transform(X_test_2d)

# Train 2D classifiers
clf_lin_2d = SVC(kernel='linear', C=1.0)
clf_rbf_2d = SVC(kernel='rbf',    C=10.0, gamma='scale')

clf_lin_2d.fit(X_train_2d_sc, y_train_ng)
clf_rbf_2d.fit(X_train_2d_sc, y_train_ng)

acc_lin_2d = accuracy_score(y_test_ng, clf_lin_2d.predict(X_test_2d_sc))
acc_rbf_2d = accuracy_score(y_test_ng, clf_rbf_2d.predict(X_test_2d_sc))

print(f'2D LinearSVC accuracy: {acc_lin_2d:.4f}')
print(f'2D RBF SVC accuracy:   {acc_rbf_2d:.4f}')

# Plot decision boundaries
x1r = np.linspace(X_train_2d_sc[:, 0].min()-0.3, X_train_2d_sc[:, 0].max()+0.3, 300)
x2r = np.linspace(X_train_2d_sc[:, 1].min()-0.3, X_train_2d_sc[:, 1].max()+0.3, 300)
xx2d, yy2d = np.meshgrid(x1r, x2r)

cat_colors  = sns.color_palette('tab10', 4)
cat_cmap_bg = ListedColormap([tuple(list(c)[:3]) + (0.4,) for c in cat_colors])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, (clf, title, acc_i) in zip(axes, [
    (clf_lin_2d, 'Linear SVM (2D SVD projection)', acc_lin_2d),
    (clf_rbf_2d, 'RBF SVM (2D SVD projection)',    acc_rbf_2d),
]):
    Z = clf.predict(np.c_[xx2d.ravel(), yy2d.ravel()]).reshape(xx2d.shape)
    ax.contourf(xx2d, yy2d, Z, alpha=0.35, cmap='tab10')
    ax.contour(xx2d, yy2d, Z, colors='k', linewidths=1.0, alpha=0.7)

    for cls_idx, (cat, color) in enumerate(zip(actual_cats, cat_colors)):
        m = y_train_ng == cls_idx
        ax.scatter(X_train_2d_sc[m, 0], X_train_2d_sc[m, 1],
                   c=[color], label=cat.split('.')[-1],
                   edgecolors='k', linewidths=0.3, s=20, alpha=0.6)

    ax.set_xlabel('SVD Component 1 (2D projection)')
    ax.set_ylabel('SVD Component 2 (2D projection)')
    ax.set_title(f'{title}\nAcc (2D)={acc_i:.4f}', fontweight='bold')
    ax.legend(fontsize=9, loc='upper right')

plt.suptitle('Decision Boundaries on 2D SVD Projection of 20 Newsgroups\n'
             '(Note: real classifier uses 10,000 features — 2D is illustrative)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 6.2: Why Linear Wins on Text (Analysis) ──────────────────────────
print('Why does LinearSVC outperform RBF SVM on 20 Newsgroups TF-IDF?')
print('=' * 65)
print()
print('1. HIGH DIMENSIONALITY (p=10,000 >> n=2,800 training docs):')
print('   - In very high-dimensional spaces, data tends to be linearly separable')
print('   - The Curse of Dimensionality: RBF kernels cannot find smooth non-linear')
print('     boundaries that generalize when p >> n')
print()
print('2. SPARSITY:')
print('   - TF-IDF vectors are sparse (< 1% non-zero entries per document)')
print('   - RBF kernel computes ||x_i - x_j||^2 between sparse vectors')
print('   - All sparse vectors are approximately equidistant → Gram matrix nearly uniform')
print('   - Linear kernel computes x_i · x_j → works correctly for sparse vectors')
print()
print('3. INFORMATION LOSS FROM TruncatedSVD:')
print('   - SVD reduces 10,000 dims → 100 dims for RBF SVM')
print('   - This loses discriminative information')
print('   - LinearSVC operates on all 10,000 dimensions directly')
print()

# Show explained variance
svd100 = TruncatedSVD(n_components=100, random_state=42)
svd100.fit(X_train_tfidf)
var_explained = svd100.explained_variance_ratio_.sum()
print(f'4. SVD (100 components) explains only {var_explained:.1%} of TF-IDF variance')
print(f'   → {1-var_explained:.1%} of discriminative information is LOST before RBF training!')

In [ ]:
# ── Cell 6.3: Borderline Documents — Near Decision Boundary in 2D ─────────
# Find test docs closest to the 2D LinearSVC decision boundary
df_scores_2d = clf_lin_2d.decision_function(X_test_2d_sc)
margin_2d    = np.max(df_scores_2d, axis=1) - np.sort(df_scores_2d, axis=1)[:, -2]
borderline   = np.argsort(margin_2d)[:4]

fig, ax = plt.subplots(figsize=(10, 7))

Z = clf_lin_2d.predict(np.c_[xx2d.ravel(), yy2d.ravel()]).reshape(xx2d.shape)
ax.contourf(xx2d, yy2d, Z, alpha=0.2, cmap='tab10')
ax.contour(xx2d, yy2d, Z, colors='k', linewidths=1.2)

for cls_idx, (cat, color) in enumerate(zip(actual_cats, cat_colors)):
    m = y_test_ng == cls_idx
    ax.scatter(X_test_2d_sc[m, 0], X_test_2d_sc[m, 1],
               c=[color], label=cat.split('.')[-1],
               edgecolors='k', linewidths=0.3, s=25, alpha=0.6)

# Highlight borderline docs
ax.scatter(X_test_2d_sc[borderline, 0], X_test_2d_sc[borderline, 1],
            s=300, facecolors='none', edgecolors='gold',
            linewidths=2.5, label='Borderline docs', zorder=6)

for k, idx in enumerate(borderline):
    ax.annotate(f'Doc {k+1}',
                (X_test_2d_sc[idx, 0], X_test_2d_sc[idx, 1]),
                textcoords='offset points', xytext=(8, 8),
                fontsize=8, color='navy', fontweight='bold')

ax.set_xlabel('SVD Component 1')
ax.set_ylabel('SVD Component 2')
ax.set_title('Test Documents Near Decision Boundary (gold circles = "borderline" docs)\n'
              'These are the support vectors of the 2D classifier', fontweight='bold')
ax.legend(fontsize=9, loc='upper right')
plt.tight_layout()
plt.show()

print('Borderline documents — ambiguous to classify:')
for k, idx in enumerate(borderline):
    true_cat = train_data.target_names[y_test_ng[idx]].split('.')[-1]
    pred_cat = train_data.target_names[clf_lin_2d.predict(X_test_2d_sc[[idx]])[0]].split('.')[-1]
    snippet  = test_data.data[idx][:120].replace('\n', ' ').strip()
    print(f'\nDoc {k+1}: True={true_cat}, Pred={pred_cat}')
    print(f'  "{snippet}..."')

## Section 7: Self-Check Questions

Test your understanding of the lab concepts. Reveal each answer after attempting it yourself.

---

**Q1.** Hard-margin SVM on 2D data has 3 support vectors. You delete the 197 non-SVs. Does retraining give the same boundary?

<details>
<summary>Answer</summary>

**Yes — exactly the same boundary** (up to numerical precision).

This is the SVM's **sparsity theorem**: the optimal hyperplane depends *only* on the support vectors. The Lagrange multipliers α_i for non-SVs are exactly zero in the dual formulation, so those points contribute nothing to w = Σ α_i y_i x_i. Removing them doesn't change w or b.

We verified this in Cell 1.3: the cosine similarity between weight vectors is ~1.0 and test accuracy is identical. This is not an approximation — it's an exact mathematical property of the SVM solution.

**Practical implication:** SVMs store only support vectors at test time → memory efficient for inference.
</details>

---

**Q2.** C=0.01 gives 150 SVs; C=100 gives 8 SVs. Which model is likely to generalize better? How does margin width differ?

<details>
<summary>Answer</summary>

**Neither is universally better** — it depends on the true data distribution. But here is the reasoning:

**C=0.01 (150 SVs, wide margin):**
- Wide margin = strong regularization = lower model complexity
- Many slack variables allowed → tolerant of misclassification
- Risk: underfitting if classes are truly separable
- Generally better on noisy data

**C=100 (8 SVs, narrow margin):**
- Narrow margin = low regularization = high model complexity
- Very few points violate the margin
- Risk: overfitting to training noise (especially if outliers forced C high)
- May generalize worse on unseen data

**Margin width formula:** width = 2/||w||. With C=0.01, ||w|| is small → wide margin. With C=100, ||w|| is large → narrow margin.

**Best practice:** Use GridSearchCV to select C based on validation performance, not training accuracy.
</details>

---

**Q3.** Why does RBF kernel with γ=100 overfit on this dataset? In terms of the Gram matrix, what does high γ do?

<details>
<summary>Answer</summary>

**Geometric explanation:**
The RBF kernel is K(x_i, x_j) = exp(-γ||x_i - x_j||²). With γ=100, even small distances (say ||x_i - x_j||=0.1) result in K ≈ exp(-100 × 0.01) = exp(-1) ≈ 0.37. For typical inter-point distances (~1.0), K ≈ exp(-100) ≈ 0 — effectively zero.

**Gram matrix effect:**
When γ is very large, the Gram matrix K becomes approximately an identity matrix — each training point is only similar to itself, not to any other point. The SVM sees each sample as a unique, isolated point in infinite-dimensional feature space.

Consequence: the decision function can be made arbitrarily complex to memorize each training label. The boundary wraps tightly around individual training points → zero training error but massive test error (overfitting).

**Underfitting (γ=0.01):** K ≈ constant for all pairs → all points look similar → the SVM cannot distinguish classes → linear boundary only.

The sweet spot (γ ≈ 1/n_features by default with gamma='scale') gives a well-conditioned Gram matrix with meaningful similarity structure.
</details>

---

**Q4.** LinearSVC achieves 92% accuracy on 20 Newsgroups TF-IDF. RBF SVM after TruncatedSVD achieves 85%. Why does dimensionality reduction hurt the RBF SVM but wouldn't hurt a linear SVM?

<details>
<summary>Answer</summary>

**Three compounding reasons:**

1. **Information loss:** TruncatedSVD with 100 components captures only ~30-40% of the TF-IDF variance. The remaining 60-70% contains discriminative information that the RBF SVM never sees. A linear SVM operating on full 10,000 features uses all of it.

2. **Linear kernels don't need dimension reduction:** The linear kernel K(x_i, x_j) = x_i · x_j already computes an inner product in the original space. There is no 'curse of dimensionality' for linear SVMs — more features = more discriminative capacity, as long as proper regularization (C) is applied.

3. **Sparsity is lost:** TF-IDF vectors are sparse (most entries = 0). After SVD, the dense 100-dimensional representations lose this structure. RBF kernel distance calculations on dense SVD vectors are geometrically distorted compared to the original sparse space.

**Why we need SVD for RBF SVM:**
Computing the RBF Gram matrix for 10,000-dim TF-IDF takes O(n²·p) time and is dominated by random structure in high dimensions (all pairwise distances become equal). SVD brings it to a tractable, meaningful low-dimensional space — but at the cost of accuracy.
</details>

---

**Q5.** You want P(class=spam | email). SVC gives `decision_function` values, not probabilities. You use `SVC(probability=True)`. What is the computational cost, and what's the alternative?

<details>
<summary>Answer</summary>

**Computational cost of `SVC(probability=True)`:**

1. Internally runs **5-fold cross-validation** on the training data to collect out-of-sample SVM scores
2. Fits a **sigmoid (Platt) calibration model**: A·f(x) + B, where A and B are fit via maximum likelihood on the OOF scores
3. Finally trains the SVM on the **full training set** again

Total: **6 SVM fits** instead of 1 → ~6× training time. For large datasets this is prohibitive.

**Additional problems:**
- The SVM trained for calibration (4/5 of data) differs from the final SVM (full data) → score distributions may not match → sigmoid fit is approximate
- Platt probabilities often cluster near 0 and 1 (over-confident)

**Better alternatives:**
1. **`CalibratedClassifierCV(svc, method='isotonic')`**: Non-parametric calibration, doesn't assume sigmoid shape, often better calibrated
2. **Logistic Regression**: Natively outputs well-calibrated probabilities; comparable accuracy to LinearSVC on text; much simpler
3. **Temperature scaling** (post-hoc): divide decision scores by a temperature T learned on a held-out set

**Recommendation:** If you need probabilities, use Logistic Regression unless you have a specific reason to prefer SVM.
</details>